# End-to-End Gemma 4 Train + Push (PhysioNet Enhanced)


In [ ]:
!pip -q install -U huggingface_hub transformers peft accelerate trl bitsandbytes datasets sentencepiece pandas yaml
!git clone https://github.com/hssling/criticalcare-copilot.git


### 1. Setup Secrets & Download Clinical Data

**Required Kaggle Secrets:**
1. `HF_HUB_TOKEN`: Your HuggingFace Write Token.
2. `PHYSIONET_USER`: Your PhysioNet email (e.g., hssling@yahoo.com).
3. `PHYSIONET_PASS`: Your PhysioNet password.

Go to **Add-ons -> Secrets** to set these.

In [ ]:
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

os.chdir('/kaggle/working/criticalcare-copilot')
user_secrets = UserSecretsClient()

try:
    pn_user = user_secrets.get_secret('PHYSIONET_USER')
    pn_pass = user_secrets.get_secret('PHYSIONET_PASS')
    os.environ['HF_HUB_TOKEN'] = user_secrets.get_secret('HF_HUB_TOKEN')
except:
    print('ERROR: Please add PHYSIONET_USER, PHYSIONET_PASS, and HF_HUB_TOKEN to Kaggle Secrets.')
    pn_user, pn_pass = None, None

if pn_user and pn_pass:
    print('Downloading clinical metadata from PhysioNet...')
    !mkdir -p data/raw/mimiciv/hosp data/raw/mimiciv/icu
    for f in ['hosp/patients.csv.gz', 'hosp/admissions.csv.gz', 'icu/icustays.csv.gz']:
        !wget -q --user {pn_user} --password {pn_pass} -O data/raw/mimiciv/{f} https://physionet.org/content/mimiciv/2.2/{f}
    
    !mkdir -p data/raw/eicu
    !wget -q --user {pn_user} --password {pn_pass} -O data/raw/eicu/patient.csv.gz https://physionet.org/content/eicu-crd/2.0/patient.csv.gz
    
    os.environ['MIMIC_IV_ROOT'] = str(Path('data/raw/mimiciv').absolute())
    os.environ['EICU_ROOT'] = str(Path('data/raw/eicu').absolute())
    
    print('Building JSONL datasets...')
    !python -m data.builders.build_mimic_cases --out data/processed/train.jsonl --limit 5000
    !python -m data.builders.build_eicu_cases   --out data/processed/valid.jsonl --limit 500


### 2. Fine-Tuning
Running the hardened QLoRA pipeline.

In [ ]:
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
print('Starting fine-tuning...')
!python training/scripts/train_sft.py --config training/configs/gemma4_e4b_qlora.yaml


### 3. Export & Push
Merging weights and uploading to HuggingFace.

In [ ]:
print('Pushing to HuggingFace...')
os.environ['HF_ORG'] = 'hssling'
os.environ['HF_MODEL_REPO'] = 'criticalcare-copilot'

!python training/scripts/merge_lora.py --base google/gemma-4-e4b-it --adapter /kaggle/working/checkpoints/gemma4_e4b_qlora --out /kaggle/working/checkpoints/merged
!python hf/publish_model.py --path /kaggle/working/checkpoints/merged --repo $HF_ORG/$HF_MODEL_REPO
